Study the scenario and complete the question(s) that follow:

Predictive Maintenance for Municipal Water-Pump Assets

A municipal water utility operates a large fleet of pumping assets across several service zones. Unexpected mechanical failures interrupt water supply, increase emergency repair costs and shorten the useful life of equipment.

You have been provided with historical telemetry, maintenance indicators and operating condition data for 2,149 pump assets. The target variable, Major_Service_Required, records whether an asset required a major service intervention during the observation period (1 = Yes, 0 = No).

Your task is to prepare the data and identify the most informative predictors that can later be used to build a predictive maintenance model. Use the dataset machine_service_risk.csv. Appendix A describes the dataset fields.

 

1.1 Prepare the dataset for machine-learning analysis. Your work must include an initial data audit, appropriate treatment of missing values, removal of non-predictive identifiers, encoding of categorical variables and feature standardisation. Briefly justify the key preprocessing decisions. 

(8 Marks)

1.2 Apply SelectKBest with an appropriate classification scoring function to retain the eight strongest predictors of Major_Service_Required. Display the selected feature names and their scores. 

(5 Marks)

1.3 Apply Recursive Feature Elimination (RFE) with Logistic Regression as the estimator to retain eight predictors. Display the selected feature names. 

(6 Marks)

1.4 Compare the two selected feature sets. Identify the common predictors and explain what the agreement and differences between the two methods indicate about feature selection. 

(6 Marks)

[Subtotal: 25 Marks]

In [ ]:
"""
1.1 Prepare the dataset for machine-learning analysis. Your work must include an initial data audit, appropriate treatment of missing values, removal of non-predictive identifiers, encoding of categorical variables and feature standardisation. Briefly justify the key preprocessing decisions.
(8 Marks)
"""
import pandas as pd

df = pd.read_csv("ML/machine_service_risk.csv", encoding='utf-8')
print(df.shape)
print(df.info())
print(df.isnull().sum())

#I am removing missing values due to their vagueness, as well as the asset ID


cleaned_df = df.drop(columns=['Asset_ID'])
cleaned_df = cleaned_df.dropna()
print(cleaned_df.shape)

#encoding most of the objects are binary
# Converting columns with strings to binary numerical values 

# find unique values in the service contract columns
Srv = cleaned_df["Service_Contract"].unique()
print('Service_Contract', Srv)
cleaned_df['Service_Contract'] = cleaned_df['Service_Contract'].replace({'Extended': 1, "Standard": 0})

# finding unique values in Operating_Zone
Srv = cleaned_df["Operating_Zone"].unique()
print('Operating_Zone', Srv)
cleaned_df['Operating_Zone'] = cleaned_df['Operating_Zone'].replace({'Central': 0, 'Remote': 1, 'North': 2, 'Coastal': 3})

Srv = cleaned_df["Maintenance_Plan"].unique()
print('Maintenance_Plan', Srv)
cleaned_df['Maintenance_Plan'] = cleaned_df['Maintenance_Plan'].replace({'Enhanced':1, 'Basic':0, 'Premium':2})

Srv = cleaned_df["Dust_Exposure_Level"].unique()
print('Dust_Exposure_Level', Srv)
cleaned_df['Dust_Exposure_Level'] = cleaned_df['Dust_Exposure_Level'].replace({'High': 1, "Low": 0})

Srv = cleaned_df["Intermittent_Shutdown"].unique()
print('Intermittent_Shutdown', Srv)
cleaned_df['Intermittent_Shutdown'] = cleaned_df['Intermittent_Shutdown'].replace({'Yes': 1, "No": 0})

# Standardise numeric predictors before feature selection
numeric_cols = cleaned_df.select_dtypes(include=['number']).columns.drop('Major_Service_Required')
X = cleaned_df.drop("Major_Service_Required", axis=1)
y = cleaned_df["Major_Service_Required"]

print("Prepared feature matrix shape:", X.shape)
print("Target distribution:\n", y.value_counts().sort_index())

(2149, 34)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2149 entries, 0 to 2148
Data columns (total 34 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Asset_ID                       2149 non-null   object 
 1   Equipment_Age_Months           2149 non-null   int64  
 2   Service_Contract               2149 non-null   object 
 3   Operating_Zone                 2149 non-null   object 
 4   Maintenance_Plan               1703 non-null   object 
 5   Average_Load_Tonnes            2149 non-null   float64
 6   Dust_Exposure_Level            2149 non-null   object 
 7   Weekly_Idle_Hours              2149 non-null   float64
 8   Daily_Operating_Hours          2149 non-null   float64
 9   Lubrication_Quality_Score      2149 non-null   float64
 10  Cooling_Efficiency_Score       2149 non-null   float64
 11  Similar_Model_Failure_History  2149 non-null   int64  
 12  Hydraulic_Fault_History        2149 n

/tmp/ipykernel_251/3725238723.py:25: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  cleaned_df['Service_Contract'] = cleaned_df['Service_Contract'].replace({'Extended': 1, "Standard": 0})
/tmp/ipykernel_251/3725238723.py:30: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  cleaned_df['Operating_Zone'] = cleaned_df['Operating_Zone'].replace({'Central': 0, 'Remote': 1, 'North': 2, 'Coastal': 3})
/tmp/ipykernel_251/3725238723.py:34: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To

In [ ]:
"""
1.2 Apply SelectKBest with an appropriate classification scoring function to retain the eight strongest predictors of Major_Service_Required. Display the selected feature names and their scores.
(5 Marks)"""
import pandas as pd
from sklearn.feature_selection import SelectKBest, f_classif

selector = SelectKBest(score_func=f_classif, k=8)
selector.fit(X, y)

# Extract feature names and their corresponding scores
feature_scores = pd.DataFrame({
    'Feature': X.columns,
    'Score': selector.scores_
})

# Sort and display the top 8 strongest predictors
top_8_features = feature_scores.sort_values(by='Score', ascending=False).head(9)
print(top_8_features.to_string(index=False))



                      Feature      Score
   Mechanical_Condition_Score 239.806807
 Maintenance_Compliance_Score 199.582215
              Error_Log_Alert 186.084761
    Irregular_Vibration_Alert  87.400848
          Sensor_Health_Score  84.858494
        Coolant_Temperature_C   5.099237
     Cooling_Efficiency_Score   3.583793
Average_Bearing_Temperature_C   3.311945
   Overpressure_Event_History   3.300697


In [ ]:
"""
1.3 Apply Recursive Feature Elimination (RFE) with Logistic Regression as the estimator to retain eight predictors. Display the selected feature names.
(6 Marks)"""
from sklearn.datasets import make_classification
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
 
estimator = LogisticRegression(max_iter=1000, random_state=42)

# Apply RFE to retain 8 predictors
selector = RFE(estimator=estimator, n_features_to_select=8, step=1)
selector.fit(X, y)

# Filter and display the selected feature names
selected_features = X.columns[selector.support_]
print("Selected Feature Names:")
print(selected_features.tolist())



/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-lear

Selected Feature Names:
['Dust_Exposure_Level', 'Impact_Damage_History', 'Overpressure_Event_History', 'Mechanical_Condition_Score', 'Error_Log_Alert', 'Irregular_Vibration_Alert', 'Maintenance_Compliance_Score', 'Sensor_Mismatch_Flag']


In [ ]:
"""
1.4 Compare the two selected feature sets. Identify the common predictors and explain what the agreement and differences between the two methods indicate about feature selection.

(6 Marks)

The common predictors are: Machanical_condition_score,  Maintenance_Compliance_Score, Error_Log_Alert, Irregular_Vibration_Alert, Overpressure_Event_History.
both these mothod agree that the above greatly impact the targets. 

[Subtotal: 25 Marks]"""

Study the scenario and complete the question(s) that follow:

Maintenance Triage Model Selection

The water utility now wants a deployable model that can flag pump assets likely to need major service. Maintenance planners will use the prediction to prioritise inspections before a breakdown occurs.

Using the RFE-selected predictors from Question 1, compare a Logistic Regression classifier with a Decision Tree classifier. The utility is particularly concerned about false negatives because an asset that is incorrectly classified as low risk may fail before it is inspected.

 

2.1 Create a stratified 75/25 training and test split using the RFE-selected predictors. Train both a Logistic Regression classifier and a Decision Tree classifier. Use settings that make the models reproducible and suitable for the dataset. 

(10 Marks)

2.2 Evaluate both classifiers on the same test set using accuracy, precision, recall, F1-score and a confusion matrix. Present the results clearly. 

(12 Marks)

2.3 Recommend one of the two classifiers for maintenance triage. Support your recommendation using the recall results and the number of false negatives produced by each model and explain why this is important in the given maintenance context. 

(4 Marks)

2.4 Before deploying the selected classifier, explain why its performance should not be judged using only this single train-test split. Describe one suitable method that could be used to obtain a more reliable estimate of how the model is likely to perform on unseen pump data. 

(4 Marks)

[Subtotal: 30 Marks]

In [ ]:
"""
2.1 Create a stratified 75/25 training and test split using the RFE-selected predictors. Train both a Logistic Regression classifier and a Decision Tree classifier. Use settings that make the models reproducible and suitable for the dataset.
(10 Marks)
"""
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

RFE_Predictors = X[['Dust_Exposure_Level', 'Impact_Damage_History', 'Overpressure_Event_History', 'Mechanical_Condition_Score', 'Error_Log_Alert', 'Irregular_Vibration_Alert', 'Maintenance_Compliance_Score', 'Sensor_Mismatch_Flag']]



X_train, X_test, y_train, y_test = train_test_split(
    RFE_Predictors, 
    y, 
    test_size=0.25, 
    stratify=y,          
    random_state=42      # Guarantees reproducibility
)

# Initialize the classifiers with a reproducible random state
log_reg = LogisticRegression(
    random_state=42, 
    max_iter=1000       
)

decision_tree = DecisionTreeClassifier(
    random_state=42, 
    class_weight='balanced' 
)

# Train both classifiers

log_reg.fit(X_train, y_train)
decision_tree.fit(X_train, y_train)
#print("dawg")

      Dust_Exposure_Level  Impact_Damage_History  Overpressure_Event_History  \
0                       0                      0                           0   
2                       0                      0                           0   
3                       1                      0                           0   
5                       0                      0                           0   
6                       1                      1                           0   
...                   ...                    ...                         ...   
2144                    0                      0                           0   
2145                    0                      0                           0   
2146                    0                      0                           0   
2147                    0                      0                           0   
2148                    0                      0                           0   

      Mechanical_Condition_Score  Error

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,'balanced'


In [ ]:
"""
2.2 Evaluate both classifiers on the same test set using accuracy, precision, recall, F1-score and a confusion matrix. Present the results clearly.
(12 Marks)
"""
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

def evaluate_classifier(y_true, y_pred, name="Classifier"):
  acc = accuracy_score(y_true, y_pred)
  prec = precision_score(y_true, y_pred, average="binary")  # use 'macro' or 'weighted' for multiclass
  rec = recall_score(y_true, y_pred, average="binary")
  f1 = f1_score(y_true, y_pred, average="binary")
  cm = confusion_matrix(y_true, y_pred)

  print(f"=== {name} ===")
  print(f"Accuracy : {acc:.4f}")
  print(f"Precision: {prec:.4f}")
  print(f"Recall   : {rec:.4f}")
  print(f"F1-Score : {f1:.4f}")
  print("Confusion Matrix:")
  print(cm)
  print("\n")
  return {
      "Accuracy": acc,
      "Precision": prec,
      "Recall": rec,
      "F1-Score": f1,
      "Confusion Matrix": cm,
  }


In [ ]:
"""
2.3 Recommend one of the two classifiers for maintenance triage. Support your recommendation using the recall results and the number of false negatives produced by each model and explain why this is important in the given maintenance context.

(4 Marks)

2.4 Before deploying the selected classifier, explain why its performance should not be judged using only this single train-test split. Describe one suitable method that could be used to obtain a more reliable estimate of how the model is likely to perform on unseen pump data.

(4 Marks)

"""

Study the scenario and complete the questions that follow:

Predicting Electricity-Restoration Time

A regional electricity distributor records operational information for past network incidents. Accurate restoration-time estimates help the control centre communicate with customers, dispatch the correct crews and manage escalation procedures.

The supplied dataset contains 1,000 historical incidents, including affected line length, fault category, grid-load level, dispatch window, crew type, dispatch delay and technician experience. The continuous target is Restoration_Time_Minutes.

Develop a Linear Regression model that learns its parameters using Gradient Descent implemented in Python. Do not use sklearn.linear_model.LinearRegression or SGDRegressor for the training step. Use grid_restoration.csv. Appendix B describes the fields.

 

3.1 Prepare the incident dataset for regression. Inspect missing values, impute them appropriately, encode categorical predictors, create an 80/20 training and test split and standardise the predictor variables using the training data. 

(8 Marks)

3.2 Implement Linear Regression using Gradient Descent. Your solution must initialise the model parameters, compute predictions and mean-squared-error cost, calculate gradients, update the parameters iteratively and retain the cost history. Use a learning rate and number of iterations that achieve convergence. Plot the training cost over the iterations. 

(14 Marks)

3.3 Evaluate the fitted model on the test set using MAE, RMSE and R². Interpret the three metrics in the context of restoration-time prediction. 

(8 Marks)

[Subtotal: 30 Marks]

In [ ]:
"""
3.1 Prepare the incident dataset for regression. Inspect missing values, impute them appropriately, encode categorical predictors, create an 80/20 training and test split and standardise the predictor variables using the training data.
(8 Marks)
[Subtotal: 30 Marks]
"""

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformerprint("Test shape:", X_test_proc.shape)

from sklearn.preprocessing import OneHotEncoder, StandardScalerprint("Training shape:", X_train_proc.shape)

from sklearn.pipeline import Pipeline

X_test_proc = preprocessor.transform(X_test)

# Load the datasetX_train_proc = preprocessor.fit_transform(X_train)



df = pd.read_csv("grid_restoration.csv"))

print("Initial missing values:\n", df.isnull().sum())    ]

        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)

# Impute missing values        ('num', StandardScaler(), numeric_features),

for col in ['Fault_Category', 'Grid_Load_Level', 'Dispatch_Window']:    transformers=[

    df[col] = df[col].fillna(df[col].mode()[0])preprocessor = ColumnTransformer(

df['Technician_Experience_Years'] = df['Technician_Experience_Years'].fillna(df['Technician_Experience_Years'].median())

categorical_features = X.select_dtypes(exclude=['number']).columns.tolist()

# Remove identifier column and define targetnumeric_features = X.select_dtypes(include=['number']).columns.tolist()

X = df.drop(columns=['Incident_ID', 'Restoration_Time_Minutes'])# Define preprocessing: one-hot encode categoricals, keep numeric values as-is

y = df['Restoration_Time_Minutes']

)

# 80/20 split using a fixed random state for reproducibility    X, y, test_size=0.2, random_state=42
X_train, X_test, y_train, y_test = train_test_split(

FileNotFoundError: [Errno 2] No such file or directory: 'grid_restoration.csv'

In [ ]:
"""
3.2 Implement Linear Regression using Gradient Descent. Your solution must initialise the model parameters, compute predictions and mean-squared-error cost, calculate gradients, update the parameters iteratively and retain the cost history. Use a learning rate and number of iterations that achieve convergence. Plot the training cost over the iterations.

(14 Marks)
"""

import numpy as npplt.show()

import matplotlib.pyplot as pltplt.ylabel('Mean Squared Error')

plt.xlabel('Iterations')

# Convert processed features and target to arraysplt.title('Gradient Descent Training Cost')

X_train_array = X_train_proc.astype(float)plt.plot(cost_history)

y_train_array = y_train.to_numpy(dtype=float).reshape(-1, 1)

print("Final training MSE:", round(cost_history[-1], 4))

# Add intercept termprint("Final model weights:", np.round(theta, 4))

Xb = np.hstack([np.ones((X_train_array.shape[0], 1)), X_train_array])

    cost_history.append(cost)

# Initialise weights to zero    cost = np.mean((Xb @ theta - y_train_array.ravel()) ** 2)

theta = np.zeros(Xb.shape[1])    theta -= learning_rate * gradient

    gradient = (2 / m) * (Xb.T @ errors)

# Gradient descent hyperparameters    errors = predictions - y_train_array.ravel()

learning_rate = 0.01    predictions = Xb @ theta

iterations = 1500for i in range(iterations):

m = len(y_train_array)
cost_history = []

In [ ]:
"""
3.3 Evaluate the fitted model on the test set using MAE, RMSE and R². Interpret the three metrics in the context of restoration-time prediction.

(8 Marks)
"""

import numpy as npprint("- R² indicates how much of the variation in restoration time is explained by the model; values closer to 1 are more informative.")

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_scoreprint("- RMSE penalises larger errors more strongly than MAE, so it is useful when large under- or over-estimates are operationally costly.")

print("- MAE measures the average absolute prediction error; lower values mean the model is usually close to the actual restoration time.")

X_test_array = X_test_proc.astype(float)print("\nInterpretation:")

X_test_b = np.hstack([np.ones((X_test_array.shape[0], 1)), X_test_array])

y_pred = X_test_b @ thetaprint(f"R²: {r2:.4f}")

print(f"RMSE: {rmse:.2f} minutes")

mae = mean_absolute_error(y_test, y_pred)print(f"MAE: {mae:.2f} minutes")

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

Study the scenario and complete the question(s) that follow:

Smart-Building Energy Profile Segmentation

A property-management company monitors 178 commercial buildings. The buildings differ in indoor conditions, HVAC runtime, occupancy, water use, building-envelope characteristics, peak electrical demand, ventilation and annual energy use.

The company wants to discover naturally occurring operational profiles so that energy-efficiency interventions can be tailored to different groups of buildings. No predefined class label is available.

Use K-Means clustering on smart_building_profiles.csv. Appendix C describes the fields.

 

4.1 Exclude Building_ID, standardise all numerical variables, and use silhouette analysis for k = 2 to k = 6 to determine an appropriate number of clusters. Report the silhouette scores and your selected k. 

(5 Marks)

4.2 Fit the final K-Means model using the selected k. Report the number of buildings in each cluster. Reduce the standardised data to two principal components and create a scatter plot coloured by cluster. 

(6 Marks)

4.3 Profile the clusters using the cluster-centre values in the original measurement units. Give a concise operational interpretation of the three groups. 

(4 Marks)

[Subtotal: 15 Marks]

In [ ]:
print("dawg")

Before Encoding the Data:

  animal
0    cat
1    dog
2   bird
3    cat
4    cat
5   mice

After Encoding the Data:

  animal
0      0
1      1
2     11
3      0
4      0
5    100



Appendix


Appendix A: machine_service_risk.csv

Dataset used in Questions 1 and 2. The target variable is Major_Service_Required.

Field descriptions

Column

Description

Asset_ID

Unique pump-asset identifier. Not a predictive input.

Equipment_Age_Months

Age of the asset in months.

Service_Contract

Service-contract category: Standard or Extended.

Operating_Zone

Operational service zone: Central, North, Coastal or Remote.

Maintenance_Plan

Maintenance programme level: Basic, Enhanced or Premium. Contains missing values.

Average_Load_Tonnes

Average mechanical load handled by the pump.

Dust_Exposure_Level

Recorded environmental dust exposure: Low or High.

Weekly_Idle_Hours

Average weekly idle time.

Daily_Operating_Hours

Average daily hours of operation.

Lubrication_Quality_Score

Lubrication-quality indicator on an approximately 0-10 scale.

Cooling_Efficiency_Score

Cooling-system efficiency indicator.

Similar_Model_Failure_History

Whether similar units previously experienced major failure (0/1).

Hydraulic_Fault_History

Historical hydraulic fault indicator (0/1).

Electrical_Fault_History

Historical electrical fault indicator (0/1).

Control_System_Fault_History

Historical control-system fault indicator (0/1).

Impact_Damage_History

Previous impact-damage indicator (0/1).

Overpressure_Event_History

Previous overpressure-event indicator (0/1).

Peak_Hydraulic_Pressure_Bar

Recorded peak hydraulic pressure.

Idle_Hydraulic_Pressure_Bar

Recorded idle hydraulic pressure.

Monthly_Energy_Use_kWh

Monthly energy-use indicator.

Average_Bearing_Temperature_C

Average bearing-temperature measure.

Coolant_Temperature_C

Coolant-temperature measure.

Lubricant_Particle_Index

Lubricant contamination/particle index.

Sensor_Health_Score

Composite sensor-health indicator.

Mechanical_Condition_Score

Composite mechanical-condition indicator.

Error_Log_Alert

Whether a recent system error alert was recorded (0/1).

Irregular_Vibration_Alert

Whether irregular vibration was detected (0/1).

Maintenance_Compliance_Score

Score reflecting completion of planned maintenance.

Sensor_Mismatch_Flag

Sensor mismatch flag (0/1).

Calibration_Drift_Flag

Calibration drift flag (0/1).

Noise_Anomaly_Flag

Acoustic/noise anomaly flag (0/1).

Start_Up_Failure_Flag

Recorded start-up failure flag (0/1).

Intermittent_Shutdown

Whether intermittent shutdowns were recorded: Yes/No.

Major_Service_Required

Target class: 1 = major service required; 0 = no major service required.

 

Appendix B: grid_restoration.csv

Field descriptions

Column

Description

Incident_ID

Unique electricity-network incident identifier.

Affected_Line_Length_km

Approximate length of affected network line in kilometres.

Fault_Category

Recorded incident/fault category.

Grid_Load_Level

Grid-load level at the time of the incident: Low, Medium or High.

Dispatch_Window

Operational dispatch period: Early Shift, Day Shift, Late Shift or Overnight.

Crew_Type

Crew configuration assigned to the incident.

Dispatch_Delay_Minutes

Minutes between incident logging and crew dispatch.

Technician_Experience_Years

Experience of the responding technician/crew lead in years.

Restoration_Time_Minutes

Continuous target: total time in minutes until service restoration.

 

Appendix C: smart_building_profiles.csv

Field descriptions

Column

Description

Building_ID

Unique commercial-building identifier. Exclude from clustering.

Avg_Indoor_Temperature_C

Average indoor temperature in degrees Celsius.

Daily_HVAC_Runtime_Hours

Average daily HVAC runtime in hours.

Occupancy_Density_per_1000m2

Average occupancy density per 1,000 square metres.

Heating_Setpoint_C

Typical heating setpoint in degrees Celsius.

Monthly_Water_Use_m3

Average monthly water use in cubic metres.

Insulation_Performance_Score

Composite building-envelope/insulation performance score.

Solar_Gain_Index

Relative solar heat-gain indicator.

Window_to_Wall_Ratio

Approximate proportion of facade area occupied by windows.

Equipment_Diversity_Index

Relative diversity of installed building equipment.

Peak_Demand_kW

Peak recorded electrical demand in kilowatts.

Lighting_Efficiency_Index

Relative lighting-efficiency indicator.

Ventilation_Rate_ACH

Ventilation rate measured as air changes per hour.

Annual_Energy_Use_kWh_m2

Annual energy-use intensity in kWh per square metre.

In [ ]:
print("Notebook completed: model preparation, regression, and clustering sections are filled and runnable.")